# Krea 2 RAW Image Generation with Multiple LoRAs

Test the local Krea 2 RAW checkpoint with zero, one, or several weighted Krea2 LoRAs before integrating the workflow into Flask. Run the cells in order using the project's Python 3.11 GPU kernel. Configuration, validation, model loading, adapter setup, and generation are separate cells so you can reuse those pieces later.

Assets resolve from `Models/Model_Families/Krea2/`: checkpoints under `checkpoints/`, LoRAs under `loras/`. Absolute file paths also work. The project root anchors `.env`, relative `CACHE_LOCATION`, and output paths. The local checkpoint supplies the transformer; `krea/Krea-2-Raw` supplies its config, Qwen3-VL text encoder/tokenizer, Qwen-Image VAE, and scheduler. First use downloads these supporting components into `CACHE_LOCATION/hub` (default `Models/hub`). Accept the upstream model's access conditions and set `HUGGINGFACE_TOKEN` in the project `.env` if required.

This is RAW text-to-image inference with the checkpoint already present locally. For Krea 2 Turbo, use its matching repository/checkpoint, distilled pipeline config, 8 steps, and guidance disabled; simply reducing RAW's steps does not turn it into Turbo.

References: [Krea 2 RAW model card](https://huggingface.co/krea/Krea-2-Raw), [Diffusers Krea2 pipeline and single-file loading](https://huggingface.co/docs/diffusers/v0.42.0/en/api/pipelines/krea2).


### 1. Check the project environment

Use the environment provisioned by `scripts/setup.sh --dev` and select its notebook kernel. This cell checks imports without upgrading the shared runtime or replacing the CUDA wheels. If dependencies are missing, use the project setup script before restarting the kernel.


In [8]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import secrets
from time import perf_counter
from uuid import uuid4

import diffusers
import peft
import torch
import transformers
from diffusers import Krea2Pipeline, Krea2Transformer2DModel
from dotenv import dotenv_values, load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from IPython.display import display

print(f"PyTorch {torch.__version__} | Diffusers {diffusers.__version__}")
print(f"Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("Krea2 pipeline, single-file transformer loader, and LoRA support imported.")


PyTorch 2.14.1+cu132 | Diffusers 0.42.1
Transformers 5.22.0 | PEFT 0.21.2
Krea2 pipeline, single-file transformer loader, and LoRA support imported.


### 2. Configure prompts, checkpoint, and LoRA stack

Append one dictionary per LoRA in `loras`, with a unique `name`, local `weights` path, independent `scale`, and `enabled` flag. One local LoRA is configured below; uncomment the second example after supplying another real Krea2 LoRA file. Add any author-required trigger words to your prompts. Only Krea2-compatible adapters should be used.

Set `use_loras = False`, `loras = []`, or disable every entry to generate with the checkpoint alone. A scale of 0 contributes nothing; 1 is the trained strength; values above 1 are allowed. Keep a fixed seed when comparing strengths. Rerun this cell and all cells below it after edits. Strength-only changes reuse loaded adapters; changing the enabled file stack reloads adapters without reloading the base model.

The RAW model card starts at 52 steps and `guidance_scale = 3.5`. Krea2 uses `cond + guidance_scale * (cond - uncond)`; 0 disables CFG, and negative prompting applies above 0. There is no separate FLUX `true_cfg_scale` control.


In [9]:
# Find the project from either its root or a notebook's nested working directory.
project_root = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "Models" / "Model_Families").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the Image_Generator project.")
model_family = "Krea2"
family_dir = project_root / "Models" / "Model_Families" / model_family
checkpoint_root = family_dir / "checkpoints"
lora_root = family_dir / "loras"

def resolve_path(value, relative_to):
    """Resolve relative paths against an explicit root; preserve absolute paths."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else relative_to / path).resolve()


prompts = [
    "A cat holding a sign that says hello world",
    "A cinematic photograph of a mountain lake at sunrise, mist over the water, golden light",
    "A friendly robot tending a rooftop greenhouse, watercolor illustration, soft pastel colors",
    "A glowing bioluminescent forest at night, tiny fireflies, intricate fantasy concept art",
]

model_id = "krea/Krea-2-Raw"
checkpoint_file = "Krea2_Raw/krea2Raw_v10_bf16.safetensors"
height = 1024                 # Pixels; positive multiples of 16.
width = 1024
num_inference_steps = 52       # RAW starting point; reduce for previews.
guidance_scale = 3.5           # Krea2 CFG convention; 0 disables guidance.
negative_prompt = ""          # Used when guidance_scale > 0.
max_sequence_length = 512      # Prompt token budget, 1–512.
num_images_per_prompt = 1      # Larger batches use more memory.
seed = 42                     # None = random; integer = repeatable seed sequence.

use_loras = True
loras = [
    {
        "name": "verite_realism",
        "weights": "Krea2_NSFW_Lora/Verite_realism.safetensors",
        "scale": 0.8,
        "enabled": True,
    },
    # Add another actual Krea2 LoRA file to test a combined stack:
    # {"name": "style", "weights": "MyStyle/style.safetensors", "scale": 0.5, "enabled": True},
]

output_dir = "Images/Krea2"    # Relative to the project root, or absolute.
show_images = True


### 3. Validate settings and local assets

Validation finishes before downloading weights or changing an existing pipeline. Disabled LoRA entries are ignored. File fingerprints detect replacements at the same path without hashing large weight files.


In [10]:
def finite_nonnegative(value):
    return (
        not isinstance(value, bool)
        and isinstance(value, (int, float))
        and math.isfinite(value)
        and value >= 0
    )


def local_weights(value, root, label):
    if not isinstance(value, (str, Path)) or not str(value).strip():
        raise ValueError(f"Set {label} to a local .safetensors file.")
    path = resolve_path(value, root)
    if path.suffix.lower() != ".safetensors":
        raise ValueError(f"{label} must be a .safetensors file.")
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    stat = path.stat()
    return path, (str(path), stat.st_size, stat.st_mtime_ns)


if not isinstance(prompts, (list, tuple)) or not prompts:
    raise ValueError("prompts must be a nonempty list of strings.")
if any(not isinstance(prompt, str) or not prompt.strip() for prompt in prompts):
    raise ValueError("Every prompt must be a nonempty string.")
unique_prompts = list(dict.fromkeys(prompts))
if not isinstance(model_id, str) or not model_id.strip():
    raise ValueError("model_id must identify the matching Krea2 RAW component repository.")
for name, value in {"height": height, "width": width}.items():
    if type(value) is not int or value <= 0 or value % 16:
        raise ValueError(f"{name} must be a positive integer divisible by 16.")
for name, value in {
    "num_inference_steps": num_inference_steps,
    "num_images_per_prompt": num_images_per_prompt,
    "max_sequence_length": max_sequence_length,
}.items():
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} must be a positive integer.")
if max_sequence_length > 512:
    raise ValueError("max_sequence_length cannot exceed 512 in this notebook.")
if seed is not None and (type(seed) is not int or not 0 <= seed < 2**63):
    raise ValueError("seed must be None or an integer between 0 and 2**63 - 1.")
if not finite_nonnegative(guidance_scale):
    raise ValueError("guidance_scale must be a finite number >= 0.")
if not isinstance(negative_prompt, str):
    raise ValueError("negative_prompt must be a string; use an empty string to leave it blank.")
if type(show_images) is not bool or type(use_loras) is not bool:
    raise ValueError("show_images and use_loras must be True or False.")
if not isinstance(output_dir, (str, Path)) or not str(output_dir).strip():
    raise ValueError("output_dir must be a nonempty folder path.")
if not isinstance(loras, (list, tuple)):
    raise ValueError("loras must be a list of LoRA configuration dictionaries.")

checkpoint_path, checkpoint_key = local_weights(checkpoint_file, checkpoint_root, "Checkpoint")
active_loras = []
adapter_names_seen = set()
if use_loras:
    for index, item in enumerate(loras):
        if not isinstance(item, dict):
            raise ValueError(f"LoRA entry {index + 1} must be a dictionary.")
        enabled = item.get("enabled", True)
        if type(enabled) is not bool:
            raise ValueError(f"LoRA entry {index + 1}: enabled must be True or False.")
        if not enabled:
            continue
        name = item.get("name")
        if not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", name):
            raise ValueError(f"LoRA entry {index + 1}: name must contain only letters, digits, _ or -.")
        if name in adapter_names_seen:
            raise ValueError(f"Duplicate enabled LoRA name: {name!r}.")
        scale = item.get("scale", 1.0)
        if not finite_nonnegative(scale):
            raise ValueError(f"LoRA {name!r}: scale must be a finite number >= 0.")
        path, fingerprint = local_weights(item.get("weights"), lora_root, f"LoRA {name!r}")
        active_loras.append({"name": name, "path": path, "scale": float(scale), "fingerprint": fingerprint})
        adapter_names_seen.add(name)

# Exclude strengths so changing only a scale does not reload the adapter files.
lora_stack_key = tuple((item["name"], item["fingerprint"]) for item in active_loras)
output_root = resolve_path(output_dir, project_root)
print(f"Validated {len(unique_prompts)} prompt(s), checkpoint, and {len(active_loras)} active LoRA(s).")


Validated 4 prompt(s), checkpoint, and 1 active LoRA(s).


### 4. Load or reuse the Krea2 RAW pipeline

This project uses an NVIDIA Thor with shared system/GPU memory. Keep components on CUDA in BF16, matching the other transformer workflows. The single-file loader converts the original Krea2 checkpoint keys; the component repository provides the matching architecture/configuration. No full base transformer is downloaded from the Hub because the local transformer is supplied explicitly.

**If loading reports HTTP 401/403:** sign in at [Krea 2 RAW](https://huggingface.co/krea/Krea-2-Raw) using the account that owns your token, accept/request access, and wait until access is granted. The token must permit reads of this gated repository. Put `HUGGINGFACE_TOKEN` or `HF_TOKEN` in the project `.env` and rerun the loading cell; it re-reads the token without needing a kernel restart. A valid token alone does not grant model access. The preflight checks the small transformer config before releasing any existing pipeline or loading checkpoint weights.


In [11]:
load_dotenv(project_root / ".env")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the project's CUDA-enabled Python kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This notebook requires a CUDA device with BF16 support.")

device = torch.device("cuda:0")
dtype = torch.bfloat16
gib = 1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print(f"GPU: {torch.cuda.get_device_name(device)} | CUDA: {torch.version.cuda}")
print(f"CUDA memory: {free_bytes / gib:.1f} GiB free / {total_bytes / gib:.1f} GiB total")

cache_root = resolve_path(os.getenv("CACHE_LOCATION") or "Models", project_root)
cache_dir = cache_root / "hub"
cache_dir.mkdir(parents=True, exist_ok=True)
# Re-read the token from .env on every run, including after editing credentials.
# Do not print tokens. None lets the Hub use its saved login when no token is set.
env_values = dotenv_values(project_root / ".env")
hf_token = (
    env_values.get("HUGGINGFACE_TOKEN") or env_values.get("HF_TOKEN")
    or os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN") or None
)
load_options = {
    "cache_dir": str(cache_dir),
    "token": hf_token,
    "dtype": dtype,
    "low_cpu_mem_usage": True,
}
pipeline_config = (model_id, checkpoint_key, str(cache_dir), dtype, str(device))
if "pipe" not in globals() or globals().get("_loaded_krea2_config") != pipeline_config:
    # Check a small required config BEFORE releasing a loaded pipeline or reading
    # the 26 GB checkpoint. The local transformer does not include these assets.
    if not Path(model_id).is_dir():
        try:
            hf_hub_download(
                repo_id=model_id,
                filename="transformer/config.json",
                cache_dir=str(cache_dir),
                token=hf_token,
            )
        except HfHubHTTPError as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            if isinstance(exc, GatedRepoError) or status in (401, 403):
                raise RuntimeError(
                    f"Hugging Face denied access to {model_id} (HTTP {status}). "
                    "The local checkpoint contains only the transformer weights; "
                    "the pipeline still needs the repository's config and supporting components. "
                    f"Open https://huggingface.co/{model_id}, sign in with the account "
                    "that owns your token, and request/accept model access. Once approved, "
                    "ensure the token has read access to this gated repository. "
                    "Set HUGGINGFACE_TOKEN or HF_TOKEN in the project .env, then rerun this cell. "
                    "If access was just granted, allow time for it to take effect."
                ) from None
            raise
    _loaded_krea2_config = None
    if "pipe" in globals():
        del pipe
    gc.collect()
    torch.cuda.empty_cache()
    transformer = Krea2Transformer2DModel.from_single_file(
        str(checkpoint_path), config=model_id, subfolder="transformer", **load_options,
    ).to(device)
    try:
        pipe = Krea2Pipeline.from_pretrained(
            model_id,
            transformer=transformer,
            is_distilled=False,
            device_map="cuda",
            use_safetensors=True,
            **load_options,
        )
    finally:
        del transformer
    _loaded_krea2_config = pipeline_config
else:
    print("Reusing the loaded Krea2 pipeline.")
print(f"Checkpoint: {checkpoint_path.name}")


GPU: NVIDIA Thor | CUDA: 13.2
CUDA memory: 4.3 GiB free / 122.9 GiB total
Reusing the loaded Krea2 pipeline.
Checkpoint: krea2Raw_v10_bf16.safetensors


### 5. Apply the selected LoRAs together

Each enabled LoRA loads under its own adapter name. `set_adapters` combines all selected adapters using their independent strengths. Adapters stay separate from the base weights, so removing/disabling the stack restores checkpoint-only inference. The Krea2 loader handles both native Diffusers adapters and supported original/AI Toolkit key layouts. A failed load clears the partial stack before raising an error.


In [12]:
try:
    if getattr(pipe, "_notebook_krea2_lora_key", None) != lora_stack_key:
        # Krea2's unload method takes no FLUX-specific reset argument.
        pipe.unload_lora_weights()
        pipe._notebook_krea2_lora_key = None
        for item in active_loras:
            print(f"Loading LoRA {item['name']}: {item['path'].name}")
            pipe.load_lora_weights(
                str(item["path"].parent),
                weight_name=item["path"].name,
                adapter_name=item["name"],
                local_files_only=True,
                use_safetensors=True,
            )
    if active_loras:
        adapter_names = [item["name"] for item in active_loras]
        adapter_weights = [item["scale"] for item in active_loras]
        pipe.set_adapters(adapter_names, adapter_weights=adapter_weights)
        pipe.set_lora_device(adapter_names=adapter_names, device=device)
        for item in active_loras:
            print(f"LoRA: {item['name']} | Strength: {item['scale']}")
    else:
        print("No active LoRAs; using the checkpoint alone.")
    pipe._notebook_krea2_lora_key = lora_stack_key
except Exception as exc:
    pipe._notebook_krea2_lora_key = None
    pipe.unload_lora_weights()
    raise RuntimeError(
        "Could not configure the Krea2 LoRA stack; adapters were cleared. "
        "Check each enabled file's Krea2 compatibility and the error below."
    ) from exc

lora_settings = [
    {"name": item["name"], "weights": str(item["path"]), "scale": item["scale"]}
    for item in active_loras
]
for name in ("transformer", "text_encoder", "vae"):
    component = getattr(pipe, name)
    print(f"{name}: {component.device}, {component.dtype}")
del component


LoRA: verite_realism | Strength: 0.8
transformer: cuda:0, torch.bfloat16
text_encoder: cuda:0, torch.bfloat16
vae: cuda:0, torch.bfloat16


### 6. Generate and save images

Prompts run sequentially; `num_images_per_prompt` sets each batch size. Images go into prompt-specific folders below `Images/Krea2` by default. Each PNG gets a JSON sidecar with the actual seed, checkpoint fingerprint, complete active LoRA stack and fingerprints, generation settings, runtime versions, and elapsed batch time. Filenames are unique across reruns. A fixed seed is repeatable within the same runtime, hardware, settings, and adapter stack; it does not guarantee identical output across library/hardware changes.


In [13]:
output_root.mkdir(parents=True, exist_ok=True)
saved_paths = []
torch.cuda.reset_peak_memory_stats(device)

# Process prompts in sequence; num_images_per_prompt controls the batch size.
with torch.inference_mode():
    for prompt_index, prompt in enumerate(unique_prompts):
        # The hash distinguishes different prompts with the same sanitized name.
        slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60].rstrip("-") or "prompt"
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
        prompt_dir = output_root / f"{slug}-{prompt_hash}"
        prompt_dir.mkdir(parents=True, exist_ok=True)
        (prompt_dir / "prompt.txt").write_text(prompt, encoding="utf-8")

        # Give each image its own recorded seed, including when seeds are random.
        seeds = [
            secrets.randbelow(2**63) if seed is None
            else (seed + prompt_index * num_images_per_prompt + image_index) % (2**63)
            for image_index in range(num_images_per_prompt)
        ]
        generators = [torch.Generator(device=device).manual_seed(value) for value in seeds]
        generation_settings = {
            "height": height,
            "width": width,
            "num_inference_steps": num_inference_steps,
            "guidance_scale": guidance_scale,
            "max_sequence_length": max_sequence_length,
            "num_images_per_prompt": num_images_per_prompt,
            "negative_prompt": negative_prompt,
        }
        print(f"Prompt {prompt_index + 1}/{len(unique_prompts)}: {prompt}")
        torch.cuda.synchronize(device)
        start = perf_counter()
        images = pipe(prompt=prompt, generator=generators, **generation_settings).images
        torch.cuda.synchronize(device)
        elapsed = perf_counter() - start

        for image, image_seed in zip(images, seeds, strict=True):
            # A fresh UUID preserves prior images, even when rerunning the same seed.
            output_path = prompt_dir / f"krea2-seed-{image_seed}-{uuid4().hex}.png"
            image.save(output_path)
            metadata = {
                "prompt": prompt,
                "model_id": model_id,
                "checkpoint": str(checkpoint_path),
                "model_family": model_family,
                "model_variant": "raw",
                "checkpoint_fingerprint": checkpoint_key,
                "loras": [dict(item) for item in lora_settings],
                "lora_fingerprints": [item["fingerprint"] for item in active_loras],
                "seed": image_seed,
                "dtype": str(dtype),
                "device": str(device),
                "torch_version": torch.__version__,
                "diffusers_version": diffusers.__version__,
                "transformers_version": transformers.__version__,
                "peft_version": peft.__version__,
                "scheduler_class": type(pipe.scheduler).__name__,
                "scheduler_config": dict(pipe.scheduler.config),
                "batch_generation_seconds": elapsed,
                **generation_settings,
            }
            output_path.with_suffix(".json").write_text(
                json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            saved_paths.append(output_path)
            print(f"Saved {output_path}")
            if show_images:
                display(image)
        print(f"Generation: {elapsed:.1f}s for {len(images)} image(s)")

print(f"Saved {len(saved_paths)} images across {len(unique_prompts)} prompt folders.")
print(f"Peak PyTorch CUDA allocation: {torch.cuda.max_memory_allocated(device) / gib:.1f} GiB")


Prompt 1/4: A cat holding a sign that says hello world


  0%|          | 0/52 [00:00<?, ?it/s]

KeyboardInterrupt: 